# Prepare LibriSpeech dataset

Builds bronze-layer manifests from a LibriSpeech-style directory tree.

**Expected layout:**
```
<landing>/
  <speaker_id>/
    <chapter_id>/
      <id>.flac
      <id>.trans.txt
```

Works with any LibriSpeech split (`dev-clean`, `test-clean`, `train-clean-100`, etc.).

In [ ]:
import os
# os.environ["ZAK_ASR_CONFIG"] = "../asr_deepspeech/config_sandbox.yml"

from sklearn.model_selection import train_test_split
from asr_deepspeech import cfg
from asr_deepspeech.etl import LibriSpeechDataset
from gnutools import fs

In [ ]:
landing = cfg.landing   # e.g. ~/data/.../landing/LibriSpeech/dev-clean
bronze  = cfg.bronze
print(f"landing : {landing}")
print(f"bronze  : {bronze}")

In [ ]:
dataset = LibriSpeechDataset(cfg.fq)
dataset = dataset.run(landing, bronze)

In [ ]:
dataset_df = dataset.filter_duration(cfg.min_duration, cfg.max_duration)
print(f"Samples after duration filter [{cfg.min_duration}s – {cfg.max_duration}s]: {len(dataset_df)}")
dataset_df.head()

In [ ]:
os.makedirs(fs.parent(cfg.label_path), exist_ok=True)
dataset.export_labels(cfg.label_path)
print(f"Labels saved to {cfg.label_path}")

dtrain, dtest = train_test_split(dataset_df, test_size=0.1, random_state=42)
os.makedirs(fs.parent(cfg.loaders.train_manifest), exist_ok=True)
dtrain.to_csv(cfg.loaders.train_manifest, index=False)
dtest.to_csv(cfg.loaders.val_manifest, index=False)
print(f"Train: {len(dtrain)} → {cfg.loaders.train_manifest}")
print(f"Test : {len(dtest)} → {cfg.loaders.val_manifest}")